# Instructor solutions

Worked answers to `teaching/exercises/`. Keep this folder apart from the student materials when distributing.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'qvdfe').exists())
sys.path.insert(0, str(ROOT / 'src'))          # works with or without `pip install -e .`
import json
import numpy as np
import matplotlib.pyplot as plt
import qvdfe
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

def check(name, ok):
    print(('PASS  ' if ok else 'FAIL  ') + name)
    assert ok, name
RATES = qvdfe.load_rates()

## T1-E1: total delay of the two closed profiles
Newell: $W_P=bP^4/36$. The sinusoidal surplus $\lambda-\mu=A\sin(2\pi t/P)$ gives $Q=\frac{AP}{2\pi}(1-\cos(2\pi t/P))$ and $W_P=AP^2/(2\pi)$.

In [2]:
mu, P, b, A = 1500, 2.0, 100, 120
t = np.linspace(0, P, 8001)
print('Newell W_P', qvdfe.NewellQueue(mu=mu, P=P, b=b).W, ' numerical', qvdfe.closed_queue(t, qvdfe.NewellQueue(mu=mu, P=P, b=b).lam(t), mu)['W_P'])
print('sine   W_P', A * P**2 / (2 * np.pi), ' numerical', qvdfe.closed_queue(t, mu + A * np.sin(2 * np.pi * t / P), mu)['W_P'])

Newell W_P 44.44444444444444  numerical 44.44444166666888
sine   W_P 76.39437268410977  numerical 76.39436875711563


## T1-E2: amplitude that makes the peak-delay vehicle reach the finite-link allowance
$w_{t_2}=4bP^3/(81\mu)=L(1/v_q-1/v_f)$, so $b^*=81\mu L(1/v_q-1/v_f)/(4P^3)$.

In [3]:
L, vf, vq = 1.0, 60.0, 20.0
bstar = 81 * mu * qvdfe.finite_link_allowance(L, vf, vq) / (4 * P**3)
print(f'b* = {bstar:.2f} veh/h^3; check peak delay {qvdfe.NewellQueue(mu=mu, P=P, b=bstar).peak_delay*60:.4f} min = allowance {qvdfe.finite_link_allowance(L, vf, vq)*60:.4f} min')

b* = 126.56 veh/h^3; check peak delay 2.0000 min = allowance 2.0000 min


## T2-E1: queue speed at which Γ changes sign (CO₂, v_f = 60 mph)

In [4]:
from scipy.optimize import brentq
c = RATES['CO2']['coef']
vs = np.linspace(5, 59, 541); g = qvdfe.gamma_cubic(vs, 60, c)
roots = [brentq(lambda v: qvdfe.gamma_cubic(v, 60, c), vs[i], vs[i + 1]) for i in range(len(vs) - 1) if g[i] * g[i + 1] < 0]
print('sign changes of Gamma(v_q) at', [round(r, 2) for r in roots], 'mph' if roots else '(none in 5-59 mph)')
for p in qvdfe.POLLUTANTS:
    g = qvdfe.gamma_cubic(vs, 60, RATES[p]['coef']); print(p, 'Gamma range', round(g.min(), 3), 'to', round(g.max(), 3))

sign changes of Gamma(v_q) at [26.57] mph
CO2 Gamma range -4849.246 to 6813.366
NOx Gamma range -18.757 to 3.754
CO Gamma range 26.519 to 53.966
HC Gamma range 1.196 to 2.6


## T2-E2: VMT/VHT form for an episode
$E_P=\frac{r(v_f)}{v_f}{\rm VMT}+\Gamma({\rm VHT}-{\rm VMT}/v_f)$; the excess vehicle-hours ${\rm VHT}-{\rm VMT}/v_f=D\bar w_P$ are converted at $\Gamma(v_q)$, not at $r(v_f)$ or at an average-speed rate.

## T3-E1: where the increment elasticity departs most from β

In [5]:
cards = qvdfe.load_cards(); cards = cards[cards.C.notna()]
rows, seen = [], set()
for c in cards.to_dict('records'):
    key = (c['corridor'], round(c['fd'], 9), round(c['n'], 9), round(c['fp'], 9), round(c['s'], 9))
    if key in seen:          # corridor-wide calibrations repeat across periods
        continue
    seen.add(key)
    r = qvdfe.response(c, c['x_median_h'], RATES['CO2']['coef'])
    if r['valid'] and np.isfinite(r['eps_Gamma']):
        rows.append((c['corridor'], c['period'], round(float(c['n']), 3), round(r['beta'], 3), round(r['Gamma'], 1), round(float(r['eps_Gamma']), 3)))
rows.sort(key=lambda z: -abs(z[5]))
print('corridor, period, n, beta, Gamma (g/(veh h)), eps_Gamma at the card median loading (admissible cards only)')
for z in rows: print(z)
print("Large |eps_Gamma| occurs where Gamma is close to zero: the ratio x Gamma'/Gamma is then large, and it is undefined at Gamma = 0.")

corridor, period, n, beta, Gamma (g/(veh h)), eps_Gamma at the card median loading (admissible cards only)
('D7_I405_S', 'AM', 1.061, 0.532, -62.9, -33.422)
('D12_I5_N', 'AM', 1.032, 0.343, -155.1, -6.857)
('D12_I5_S', 'AM', 1.031, 0.914, -179.3, -5.354)
('D12_I5_S', 'MD', 1.036, 0.651, -1039.3, -1.221)
('D12_I5_N', 'PM', 1.007, 0.333, 523.7, 0.4)
Large |eps_Gamma| occurs where Gamma is close to zero: the ratio x Gamma'/Gamma is then large, and it is undefined at Gamma = 0.


## T3-E2: marginal emission at the reference loading, split into the average and the external part

In [6]:
from qvdfe.data import load_reference_state
ref = load_reference_state()
card = cards[(cards.corridor == 'AZ_I10_W') & (cards.period == 'MD')].iloc[0].to_dict()
r0 = qvdfe.response(card, ref['x_h'], RATES['CO2']['coef'])
for name, c in (('calibrated n', card), ('fixed service mu0 = mu(x0)', {**card, 'n': 1.0, 'fd': r0['P'] / ref['x_h']})):
    r = qvdfe.response(c, ref['x_h'], RATES['CO2']['coef'])
    print(f"{name:20s} e = {r['e_g']:.2f} g/veh, external x e'(x) = {r['external_g']:.2f} g/veh, marginal dE/dD = {r['marginal_g']:.2f} g/veh (valid={r['valid']})")
print('Both rows share e (same state at x0). Under fixed service the external part is beta * Gamma * wbar only;')
print('the calibrated card adds wbar * x dGamma/dx, the queue-state term, because v_q falls as loading rises.')

calibrated n         e = 326.24 g/veh, external x e'(x) = 51.56 g/veh, marginal dE/dD = 377.79 g/veh (valid=True)
fixed service mu0 = mu(x0) e = 326.24 g/veh, external x e'(x) = 30.15 g/veh, marginal dE/dD = 356.39 g/veh (valid=True)
Both rows share e (same state at x0). Under fixed service the external part is beta * Gamma * wbar only;
the calibrated card adds wbar * x dGamma/dx, the queue-state term, because v_q falls as loading rises.


## T4-E1: sensitivity of the band to the peak magnitudes

In [7]:
from qvdfe.data import load_reference_state
ref = load_reference_state(); L, vf, vq = ref['L_mi'], ref['vf_mph'], ref['vq_mph']
for ad, au in [(1.0, 0.67), (1.5, 1.0), (2.0, 1.5), (3.0, 2.0)]:
    lo, hi = qvdfe.transition_band(L, vf, vq, qvdfe.transition_time_s(vf - vq, ad) / 3600, qvdfe.transition_time_s(vf - vq, au) / 3600)
    print(f'({ad}, {au}) m/s^2: band {lo*60:.3f}-{hi*60:.3f} min; peak delay {ref["peak_delay_h"]*60:.3f} min admitted: {lo <= ref["peak_delay_h"] <= hi}')

(1.0, 0.67) m/s^2: band 0.470-0.856 min; peak delay 1.825 min admitted: False


(1.5, 1.0) m/s^2: band 0.314-1.421 min; peak delay 1.825 min admitted: False
(2.0, 1.5) m/s^2: band 0.220-1.764 min; peak delay 1.825 min admitted: False
(3.0, 2.0) m/s^2: band 0.157-1.992 min; peak delay 1.825 min admitted: True
